# CSP 作业：约束满足问题

| 部分 | 内容 | 待完成项 |
|------|------|----------|
| 部分 0 | 问题定义（已给出） | 0 |
| 部分 1 | `is_consistent` | 1 |
| 部分 2 | AC-3：`revise` + `ac3` | 3 |
| 部分 3 | 回溯：`forward_checking` + `select_mrv_variable` + `order_lcv_values` | 3 |
| 部分 4 | 验证（已给出，直接运行） | 0 |

> 完成全部 TODO，并确保每个检查单元在提交前都打印 OK。


---
## 部分 0：问题定义
无需修改，直接运行。


In [ ]:
# 问题 A：澳大利亚地图着色（用于 AC-3）
AUSTRALIA = {
    'variables': ['WA', 'NT', 'SA', 'Q', 'NSW', 'V', 'T'],
    'neighbors': {
        'WA':  ['NT', 'SA'],
        'NT':  ['WA', 'SA', 'Q'],
        'SA':  ['WA', 'NT', 'Q', 'NSW', 'V'],
        'Q':   ['NT', 'SA', 'NSW'],
        'NSW': ['Q', 'SA', 'V'],
        'V':   ['SA', 'NSW'],
        'T':   []
    },
    'colors': ['red', 'green', 'blue']
}

# 问题 B：自定义非对称图（用于 FC+MRV+LCV）
# 该图被设计为在 FC 之后产生不同大小的取值域，
# 从而使 MRV 和 LCV 的选择不再是显然的。
ASYMMETRIC = {
    'variables': ['A', 'B', 'C', 'D', 'E', 'F'],
    'neighbors': {
        'A': ['B', 'C', 'D'],
        'B': ['A', 'C'],
        'C': ['A', 'B', 'D', 'E'],
        'D': ['A', 'C', 'F'],
        'E': ['C', 'F'],
        'F': ['D', 'E']
    },
    'colors': ['red', 'green', 'blue']
}

def make_domains(problem):
    return {v: set(problem['colors']) for v in problem['variables']}

print('Part 0 loaded OK')
print('Australia  :', AUSTRALIA['variables'])
print('Asymmetric :', ASYMMETRIC['variables'])


Part 0 loaded OK
Australia  : ['WA', 'NT', 'SA', 'Q', 'NSW', 'V', 'T']
Asymmetric : ['A', 'B', 'C', 'D', 'E', 'F']


---
## 部分 1：约束检查

### TODO 1 / 7  --  `is_consistent`

当赋值 `var = value` 与所有已赋值邻居都不冲突时，返回 `True`。
两个节点在相邻且颜色相同的情况下发生冲突。


In [ ]:
def is_consistent(var, value, assignment, neighbors):
    """
    若 var=value 与所有已赋值邻居都兼容，则返回 True。

    参数:
        var        : 待赋值变量 (str)
        value      : 候选颜色 (str)
        assignment : 已赋值变量字典
        neighbors  : 邻接表字典
    """
    # ---------------------------------------------------------------
    # TODO 1 
    # 遍历 var 的每个邻居:
    #   如果该邻居已赋值且其颜色等于 value
    #   -> 返回 False
    # ---------------------------------------------------------------

    for neighbor in neighbors[var]:
        if neighbor in assignment and assignment[neighbor] == value:
            return False

    return True


In [ ]:
# --- 验证 TODO 1 ---
_a = {'WA': 'red', 'NT': 'green'}
assert is_consistent('SA', 'blue', _a, AUSTRALIA['neighbors']) == True
assert is_consistent('SA', 'red', _a, AUSTRALIA['neighbors']) == False
assert is_consistent('SA', 'green', _a, AUSTRALIA['neighbors']) == False
assert is_consistent('T',  'red', _a, AUSTRALIA['neighbors']) == True
print('OK: is_consistent')


OK: is_consistent


---
## 部分 2：AC-3

AC-3 通过反复裁剪变量取值域来维持弧一致性。
对弧 (Xi -> Xj)：删除 Xi 中所有在 Xj 中找不到兼容值的取值。

### TODO 2 / 7  --  `revise`


In [ ]:
def revise(domains, Xi, Xj):
    """
    从 domains[Xi] 中删除所有与 domains[Xj] 的每个值都冲突的取值。
    对图着色问题而言，当且仅当两个颜色相等时冲突。

    若 domains[Xi] 发生变化则返回 True，否则返回 False。
    """
    removed = False
    for x in list(domains[Xi]):
        # ---------------------------------------------------------------
        # TODO 2
        # 检查 domains[Xj] 中是否存在与 x 兼容的值（即不等于 x 的值）。
        # 如果不存在任何兼容值，则从 domains[Xi] 删除 x，
        # 并设置 removed = True。
        # ---------------------------------------------------------------

        if all(x == y for y in domains[Xj]):
            domains[Xi].remove(x)
            removed = True

    return removed


In [ ]:
# --- 验证 TODO 2 ---
_d = {'Xi': {'red', 'green', 'blue'}, 'Xj': {'red'}}
assert revise(_d, 'Xi', 'Xj') == True
assert 'red' not in _d['Xi']
assert _d['Xi'] == {'green', 'blue'}
_d2 = {'A': {'red', 'green'}, 'B': {'green', 'blue'}}
assert revise(_d2, 'A', 'B') == False
assert _d2['A'] == {'red', 'green'}
print('OK: revise')


OK: revise


### TODO 3 + 4 / 7  --  `ac3`

先用**所有有向弧**（每条边两个方向）初始化队列，
然后迭代处理，直到收敛或某个变量域变为空。


In [ ]:
from collections import deque

def ac3(domains, neighbors):
    """
    完整 AC-3 算法。
    若达到弧一致，返回 (True, domains)；
    若任一变量域变为空，返回 (False, None)。
    """
    # ---------------------------------------------------------------
    # TODO 3 
    # 构建初始队列：对每对相邻变量加入 (Xi, Xj)，
    # 且两个方向都要加入。
    # ---------------------------------------------------------------

    queue = deque()  # 请完善此处
    for Xi in neighbors:
        for Xj in neighbors[Xi]:   # 对于每个邻居关系
            queue.append((Xi, Xj)) # 加入弧 Xi -> Xj

    while queue:
        Xi, Xj = queue.popleft()
        if revise(domains, Xi, Xj):

            # -----------------------------------------------------------
            # TODO 4a 
            # 如果 domains[Xi] 此时为空 -> 不一致，返回 (False, None)
            # -----------------------------------------------------------

            # TODO 4a: 若 Xi 的域变为空，则失败
            if not domains[Xi]:
                return False, None

            # -----------------------------------------------------------
            # TODO 4b 
            # Xi 缩小后，Xi 的邻居可能需要重新检查。
            # 对 Xi 的每个邻居 Xk（且 Xk != Xj）重新加入 (Xk, Xi)。
            # -----------------------------------------------------------

            # TODO 4b: 将受影响的邻居 Xk 重新加入队列 (Xk, Xi)，Xk ≠ Xj
            for Xk in neighbors[Xi]:
                if Xk != Xj:
                    queue.append((Xk, Xi))

    return True, domains


In [ ]:
# --- 验证 TODO 3 + 4 ---
# 场景 1：必须将 {WA=green, V=red} 判定为不一致
_d = make_domains(AUSTRALIA)
_d['WA'] = {'green'}; _d['V'] = {'red'}
_ok, _ = ac3(_d, AUSTRALIA['neighbors'])
assert _ok == False, 'Expected inconsistency'
print('OK: ac3 scenario 1 -- inconsistency detected')

# 场景 2：仅有 {WA=red} 时应保持一致；NT 和 SA 必须去除 red
_d2 = make_domains(AUSTRALIA)
_d2['WA'] = {'red'}
_ok2, _f = ac3(_d2, AUSTRALIA['neighbors'])
assert _ok2 == True
assert 'red' not in _f['NT']
assert 'red' not in _f['SA']
print('OK: ac3 scenario 2 -- propagation correct')


OK: ac3 scenario 1 -- inconsistency detected
OK: ac3 scenario 2 -- propagation correct


---
## 部分 3：带 FC + MRV + LCV 的回溯

请实现三个辅助函数。完整的 `backtrack` 函数在后面已给出。

### TODO 5 / 7  --  `forward_checking`


In [ ]:
def forward_checking(var, value, domains, assignment, neighbors):
    """
    在赋值 var=value 后，从每个未赋值邻居的取值域中
    删除冲突颜色。

    在深拷贝上操作，不会修改原始 domains。

    返回新的 domains 字典；若某个邻居域被删空则返回 None。
    """
    new_domains = {v: set(d) for v, d in domains.items()}  # 深拷贝

    for neighbor in neighbors[var]:
        if neighbor not in assignment:
            # ---------------------------------------------------------------
            # TODO 5 
            # 从 new_domains[neighbor] 中删除 `value`。
            # 若 new_domains[neighbor] 变为空，返回 None。
            # ---------------------------------------------------------------

            # 删除与 value 冲突的颜色（即 value 本身）
            if value in new_domains[neighbor]:
                new_domains[neighbor].discard(value)
                if not new_domains[neighbor]:
                    return None            

    return new_domains


In [ ]:
# --- 验证 TODO 5 ---
_d = make_domains(ASYMMETRIC)
_r = forward_checking('A', 'red', _d, {'A': 'red'}, ASYMMETRIC['neighbors'])
assert _r is not None
assert 'red' not in _r['B']
assert 'red' not in _r['C']
assert 'red' not in _r['D']
assert 'red' in _d['B'], 'Must not modify original domains'
print('OK: forward_checking')


OK: forward_checking


### TODO 6 / 7  --  `select_mrv_variable`

**MRV**：选择当前取值域最小的未赋值变量。
若并列，则用**Degree**打破平局：优先选择受未赋值邻居约束更多的变量。


In [ ]:
def select_mrv_variable(domains, assignment, neighbors):
    """
    返回取值域最小的未赋值变量（MRV）。
    如有并列，按未赋值邻居数量打破平局，数量更多者优先。
    """
    unassigned = [v for v in domains if v not in assignment]

    # ---------------------------------------------------------------
    # TODO 6
    # 遍历 unassigned，记录当前最优变量 best。
    # 对每个变量 v，与 best 比较：
    #   - 若 v 的域更小，则更新 best = v
    #   - 若域大小相同，则比较各自未赋值邻居的数量，
    #     数量更多者胜出（Degree 启发式打破平局）
    # 返回 best。
    # ---------------------------------------------------------------

    def degree(var):
        # 计算未赋值邻居的数量
        return sum(1 for nb in neighbors[var] if nb not in assignment)

    best = unassigned[0]
    for v in unassigned[1:]:
        # MRV：域越小越好
        if len(domains[v]) < len(domains[best]):
            best = v
        elif len(domains[v]) == len(domains[best]):
            # 平局时比较未赋值邻居数（Degree 启发式）
            if degree(v) > degree(best):
                best = v
    return best


In [ ]:
# --- 验证 TODO 6 ---
_d = make_domains(AUSTRALIA)
_d['SA'] = {'red'}
for _nb in AUSTRALIA['neighbors']['SA']: _d[_nb].discard('red')
_chosen = select_mrv_variable(_d, {'SA': 'red'}, AUSTRALIA['neighbors'])
assert _chosen != 'T', f'T has the largest domain; MRV must not pick it. Got: {_chosen}'
assert len(_d[_chosen]) == 2
print(f'OK: select_mrv_variable -> {_chosen} (domain size {len(_d[_chosen])})')


OK: select_mrv_variable -> NT (domain size 2)


### TODO 7 / 7  --  `order_lcv_values`

**LCV**：按候选值排序，让“排除邻居选项最少”的值优先。


In [ ]:
def order_lcv_values(var, domains, assignment, neighbors):
    """
    按 LCV 顺序返回 domains[var] 中的取值（优先选择淘汰数最少者）。
    """
    def count_eliminations(value):
        # ---------------------------------------------------------------
        # TODO 7 
        # 对 var 的每个未赋值邻居 nb:
        # 若 `value` 在 domains[nb] 中，则计数加 1。
        # 返回总计数。
        # ---------------------------------------------------------------

        elim = 0
        for nb in neighbors[var]:
            if nb not in assignment and value in domains[nb]:
                elim += 1
        return elim

    return sorted(domains[var], key=count_eliminations)


In [ ]:
# --- 验证 TODO 7 ---
# X=red 会同时影响 Y 和 Z；X=green/blue 只影响 Y -> red 应该排在最后
_d = {'X': {'red','green','blue'}, 'Y': {'red','green','blue'}, 'Z': {'red'}}
_nb = {'X': ['Y','Z'], 'Y': ['X'], 'Z': ['X']}
_order = order_lcv_values('X', _d, {}, _nb)
assert _order[-1] == 'red', f'Red is most constraining, should be last. Got: {_order}'
print(f'OK: order_lcv_values -> {_order}  (red last)')


OK: order_lcv_values -> ['blue', 'green', 'red']  (red last)


### 完整回溯求解器（已给出，请勿修改）


In [ ]:
def backtrack(assignment, domains, problem):
    if len(assignment) == len(problem['variables']):
        return assignment
    var = select_mrv_variable(domains, assignment, problem['neighbors'])
    for value in order_lcv_values(var, domains, assignment, problem['neighbors']):
        if is_consistent(var, value, assignment, problem['neighbors']):
            assignment[var] = value
            new_domains = forward_checking(
                var, value, domains, assignment, problem['neighbors'])
            if new_domains is not None:
                result = backtrack(assignment, new_domains, problem)
                if result is not None:
                    return result
            del assignment[var]
    return None

def solve(problem):
    return backtrack({}, make_domains(problem), problem)

print('Backtrack solver loaded OK')


Backtrack solver loaded OK


---
## 部分 4：验证
无需修改。运行后与手算答案对比。

### 验证 1：求解非对称图


In [ ]:
sol = solve(ASYMMETRIC)
print('Solution:', sol)
if sol:
    ok = all(sol[v] != sol[nb]
             for v, nbs in ASYMMETRIC['neighbors'].items() for nb in nbs)
    print('All constraints satisfied:', ok)


Solution: {'C': 'red', 'A': 'blue', 'D': 'green', 'B': 'green', 'E': 'green', 'F': 'red'}
All constraints satisfied: True


### 验证 2：在 {WA=green, V=red} 上比较 FC 与 AC-3


In [ ]:
# -- 前向检查 --
print('--- Forward Checking ---')
_d_fc = make_domains(AUSTRALIA)
_afc  = {'WA': 'green'}
_d_fc = forward_checking('WA', 'green', _d_fc, _afc, AUSTRALIA['neighbors'])
_afc['V'] = 'red'
_d_fc = forward_checking('V', 'red', _d_fc, _afc, AUSTRALIA['neighbors'])
if _d_fc is None:
    print('FC detected inconsistency')
else:
    for v in AUSTRALIA['variables']:
        if v not in _afc:
            print(f'  {v}: {_d_fc[v]}{"  <- EMPTY" if not _d_fc[v] else ""}')
    print('FC did NOT detect inconsistency')

# -- AC-3 --
print('--- AC-3 ---')
_d_ac = make_domains(AUSTRALIA)
_d_ac['WA'] = {'green'}; _d_ac['V'] = {'red'}
_ok, _ = ac3(_d_ac, AUSTRALIA['neighbors'])
print('AC-3 detected inconsistency:', not _ok)

print()
print('FC  detected inconsistency?', 'Yes' if _d_fc is None else 'No')
print('AC-3 detected inconsistency?', 'Yes' if not _ok else 'No')
print('Does this match Theory Q2(d)?', 'Yes')
print('  -> FC 只向前看一步，SA 域缩减为 {blue} 但未变空，故无法察觉矛盾。')
print('  -> AC-3 持续传播弧一致性，最终将某变量域裁减为空，检测到不一致。')


--- Forward Checking ---
  NT: {'red', 'blue'}
  SA: {'blue'}
  Q: {'red', 'blue', 'green'}
  NSW: {'blue', 'green'}
  T: {'red', 'blue', 'green'}
FC did NOT detect inconsistency
--- AC-3 ---
AC-3 detected inconsistency: True

FC  detected inconsistency? No
AC-3 detected inconsistency? Yes
Does this match Theory Q2(d)? Yes
  -> FC 只向前看一步，SA 域缩减为 {blue} 但未变空，故无法察觉矛盾。
  -> AC-3 持续传播弧一致性，最终将某变量域裁减为空，检测到不一致。


### 最终检查 -- 提交前请运行


In [ ]:
print('Submission check')
print('-' * 36)

def assert_(cond): assert cond

def _ok(name, fn):
    try: fn(); print(f'  OK    {name}')
    except Exception as e: print(f'  FAIL  {name}  ({e})')

_ok('TODO 1  is_consistent', lambda: (
    (_ := {'WA':'red','NT':'green'}),
    assert_(is_consistent('SA', 'blue', _, AUSTRALIA['neighbors'])),
    assert_(not is_consistent('SA', 'red', _, AUSTRALIA['neighbors']))
))
_ok('TODO 2  revise', lambda: (
    (_ := {'A':{'red','green','blue'},'B':{'red'}}),
    assert_(revise(_, 'A', 'B')),
    assert_('red' not in _['A'])
))
_ok('TODO 3+4 ac3', lambda: (
    (_ := dict(make_domains(AUSTRALIA), **{'WA':{'green'},'V':{'red'}})),
    assert_(not ac3(_, AUSTRALIA['neighbors'])[0])
))
_ok('TODO 5  forward_checking', lambda: (
    (_ := forward_checking('A','red',make_domains(ASYMMETRIC),{'A':'red'},ASYMMETRIC['neighbors'])),
    assert_(_ is not None and 'red' not in _['B'])
))
_ok('TODO 6  select_mrv_variable', lambda: (
    (_ := dict(make_domains(AUSTRALIA), **{'SA':{'red'}})),
    [_[nb].discard('red') for nb in AUSTRALIA['neighbors']['SA']],
    assert_(select_mrv_variable(_, {'SA':'red'}, AUSTRALIA['neighbors']) != 'T')
))
_ok('TODO 7  order_lcv_values', lambda: (
    assert_(order_lcv_values(
        'X',{'X':{'red','green','blue'},'Y':{'red','green','blue'},'Z':{'red'}},
        {},{'X':['Y','Z'],'Y':['X'],'Z':['X']})[-1] == 'red'
    )
))
_ok('Full solve', lambda: (
    (_ := solve(ASYMMETRIC)),
    assert_(_ is not None),
    assert_(all(_[v] != _[nb]
        for v,nbs in ASYMMETRIC['neighbors'].items() for nb in nbs))
))


Submission check
------------------------------------
  OK    TODO 1  is_consistent
  OK    TODO 2  revise
  OK    TODO 3+4 ac3
  OK    TODO 5  forward_checking
  OK    TODO 6  select_mrv_variable
  OK    TODO 7  order_lcv_values
  OK    Full solve
